# Assignment 2 — Evaluation under animation (VASA-1 concept, implemented with LivePortrait)

**Idea (VASA-1, Microsoft 2024):** turn a single portrait into a talking-head video. VASA-1 itself is not publicly released,
so we use the open-source **LivePortrait** (Kuaishou, 2024) to animate our **test** photos with a reference talking video,
then check whether the ViT keeps predicting the same emotion on every frame.

| Step | Content |
|---|---|
| 1 | Install LivePortrait and download its weights |
| 2 | Pick test photos (from `test_sources.csv` of the main notebook) and animate them |
| 3 | Save clean clips (for evaluation) + watermarked previews labelled **SYNTHETIC** (for the app / demo video) |
| 4 | Run the trained ViT (`A2/results/vit_emotion`) on stills and on every frame → **Δmacro-F1**, consistency, emotion timeline, attention heatmaps |

Two animation settings are compared (if the installed LivePortrait supports `animation_region`):
* `pose` — only head motion is transferred; the expression of the photo is kept (label stays valid)
* `lip` — only mouth motion is transferred (talking) — a harder stress test, since the mouth carries most of the emotion

**Run this in a fresh Colab session with a T4 GPU** (Runtime → Change runtime type → T4 GPU), then Runtime → Run all. About 15–25 minutes.
The main notebook does **not** need to be re-run.

*Consent: only photos of people who agreed to take part are animated; all generated clips are labelled as synthetic.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Settings

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/Dataset-folder/dataset'
A2_DIR      = '/content/drive/MyDrive/A2'
RESULTS_DIR = f'{A2_DIR}/results'
MODEL_DIR   = f'{RESULTS_DIR}/vit_emotion'
CLIPS_DIR   = f'{A2_DIR}/clips'           # clean clips used for evaluation
PREVIEW_DIR = f'{A2_DIR}/clips_preview'   # watermarked copies for the app / demo video
LP_DIR      = '/content/LivePortrait'

EMOTIONS           = ['happy', 'sad', 'neutral']
PHOTOS_PER_EMOTION = 8          # test photos animated per emotion
N_DRIVING          = 1          # reference talking videos per photo
REGIONS            = ['pose', 'lip']
MAX_SECONDS        = 6          # clip length
FRAME_STEP         = 3          # evaluate every 3rd frame
SEED               = 42

## 1. Install LivePortrait

In [ ]:
import os, sys, re, glob, json, random, shutil, subprocess
if not os.path.exists(LP_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/KwaiVGI/LivePortrait', LP_DIR], check=True)

!pip -q install tyro rich ffmpeg-python pykalman onnxruntime-gpu imageio-ffmpeg lmdb "huggingface_hub>=0.23"
from huggingface_hub import snapshot_download
snapshot_download('KwaiVGI/LivePortrait', local_dir=f'{LP_DIR}/pretrained_weights',
                  ignore_patterns=['*.git*', 'README.md', 'docs/*'])

# start LivePortrait once; install any module it still misses (without touching Colab's torch/numpy)
PIP_NAME = {'cv2': 'opencv-python', 'yaml': 'pyyaml', 'skimage': 'scikit-image', 'ffmpeg': 'ffmpeg-python',
            'PIL': 'pillow', 'onnxruntime': 'onnxruntime-gpu', 'sklearn': 'scikit-learn'}
for _ in range(10):
    r = subprocess.run([sys.executable, 'inference.py', '-h'], cwd=LP_DIR, capture_output=True, text=True)
    if r.returncode == 0:
        break
    m = re.search(r"No module named '([^'.]+)", r.stderr)
    if not m:
        print(r.stderr[-3000:])
        raise RuntimeError('LivePortrait could not start - send a screenshot of the error above')
    pkg = PIP_NAME.get(m.group(1), m.group(1))
    print('installing missing module:', pkg)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
HELP = r.stdout

def cli_flag(name):
    for f in (f'--{name}', '--' + name.replace('_', '-')):
        if re.search(re.escape(f) + r'\b', HELP):
            return f
    return None

REGION_FLAG, OUT_FLAG = cli_flag('animation_region'), cli_flag('output_dir')
if REGION_FLAG is None:
    print('This LivePortrait version has no animation_region option -> full animation ("all") is used; '
          'the driving expression is transferred, so treat frame labels with care.')
    REGIONS = ['all']
print('LivePortrait ready | region flag:', REGION_FLAG, '| output flag:', OUT_FLAG, '| regions:', REGIONS)

## 2. Choose test photos and driving videos

In [ ]:
import pandas as pd
test_src = pd.read_csv(f'{RESULTS_DIR}/test_sources.csv')
splits   = pd.read_csv(f'{RESULTS_DIR}/splits.csv')
test_files = set(splits[(splits.split == 'test') & splits.used].file)
test_src = test_src[test_src.file.isin(test_files)]           # safety: only real test photos

parts = []
for e in EMOTIONS:
    d = test_src[test_src.label == e].drop_duplicates('source_key')
    parts.append(d.sample(min(len(d), PHOTOS_PER_EMOTION), random_state=SEED))
chosen = pd.concat(parts, ignore_index=True)
chosen['path'] = chosen.file.map(lambda f: os.path.join(DATASET_DIR, f))
display(chosen.groupby('label').size().rename('photos to animate'))

drv_all = sorted(glob.glob(f'{LP_DIR}/assets/examples/driving/*.mp4'))
drv_all = sorted(drv_all, key=lambda p: os.path.basename(p) != 'd0.mp4')   # the default talking example first
os.makedirs('/content/lp_drv', exist_ok=True)
DRIVING = []
for p in drv_all[:N_DRIVING]:
    out = f'/content/lp_drv/{os.path.splitext(os.path.basename(p))[0]}.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', p, '-t', str(MAX_SECONDS), '-an', out], check=True)
    DRIVING.append(out)
print('Driving videos:', [os.path.basename(p) for p in DRIVING])

## 3. Generate the clips

In [ ]:
import cv2, numpy as np, time
os.makedirs('/content/lp_src', exist_ok=True)
for e in EMOTIONS:
    os.makedirs(f'{CLIPS_DIR}/{e}', exist_ok=True)
    os.makedirs(f'{PREVIEW_DIR}/{e}', exist_ok=True)

def watermark(src_mp4, dst_mp4):
    cap = cv2.VideoCapture(src_mp4)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tmp = '/content/_wm.mp4'
    out = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    text = 'SYNTHETIC - AI-generated (LivePortrait, VASA-1 concept) - used with consent'
    scale = max(0.35, w / 1400)
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        cv2.rectangle(fr, (0, h - int(34 * scale / 0.5)), (w, h), (0, 0, 0), -1)
        cv2.putText(fr, text, (8, h - int(10 * scale / 0.5)), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)
        out.write(fr)
    cap.release(); out.release()
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', dst_mp4])

made, failed = [], []
t0 = time.time()
for _, row in chosen.iterrows():
    ext = os.path.splitext(row.path)[1]
    src = f'/content/lp_src/{row.source_key}{ext}'
    shutil.copy(row.path, src)
    for drv in DRIVING:
        dname = os.path.splitext(os.path.basename(drv))[0]
        for region in REGIONS:
            outdir = f'/content/lp_out/{region}'
            os.makedirs(outdir, exist_ok=True)
            cmd = [sys.executable, 'inference.py', '-s', src, '-d', drv]
            if REGION_FLAG and region != 'all':
                cmd += [REGION_FLAG, region]
            if OUT_FLAG:
                cmd += [OUT_FLAG, outdir]
            r = subprocess.run(cmd, cwd=LP_DIR, capture_output=True, text=True)
            search_dir = outdir if OUT_FLAG else f'{LP_DIR}/animations'
            res = [p for p in glob.glob(f'{search_dir}/{row.source_key}--{dname}*.mp4') if '_concat' not in p]
            if not res:   # other naming: take the newest result in the output folder
                res = sorted([p for p in glob.glob(f'{search_dir}/*.mp4') if '_concat' not in p], key=os.path.getmtime)[-1:]
            if r.returncode != 0 or not res:
                failed.append((row.source_key, region, (r.stderr or r.stdout)[-400:]))
                continue
            name = f'{row.source_key}__{dname}_{region}.mp4'
            shutil.copy(res[0], f'{CLIPS_DIR}/{row.label}/{name}')
            watermark(res[0], f'{PREVIEW_DIR}/{row.label}/{name}')
            made.append({'clip': name, 'emotion': row.label, 'source_key': row.source_key, 'region': region, 'source_path': row.path})
            os.remove(res[0])
    print(f'{len(made)} clips done ({time.time() - t0:.0f}s)')

print(f'\nGenerated {len(made)} clips, failed {len(failed)}')
for f in failed[:5]:
    print('FAILED', f[0], f[1], '\n', f[2])
made_df = pd.DataFrame(made)
made_df.to_csv(f'{RESULTS_DIR}/animation_manifest.csv', index=False)

## 4. Evaluate the trained ViT on stills vs. animated frames
* **Stills macro-F1** — ViT on the original test photos that were animated
* **Frames macro-F1** — ViT on every `FRAME_STEP`-th frame of their clips
* **Δmacro-F1 = frames − stills** (negative = the model is less reliable under animation)
* **Consistency** — share of frames that agree with the clip's majority prediction (1.0 = perfectly stable)

In [ ]:
import torch
from torchvision import transforms as T
from PIL import Image
from transformers import ViTForImageClassification
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = json.load(open(f'{MODEL_DIR}/app_config.json'))
LABELS = cfg['labels']
model = ViTForImageClassification.from_pretrained(MODEL_DIR, attn_implementation='eager').to(device).eval()
tf = T.Compose([T.Resize((cfg['img_size'], cfg['img_size'])), T.ToTensor(), T.Normalize(cfg['mean'], cfg['std'])])
lab2id = {e: i for i, e in enumerate(LABELS)}

@torch.no_grad()
def predict_pils(pils):
    out = []
    for i in range(0, len(pils), 32):
        x = torch.stack([tf(p) for p in pils[i:i + 32]]).to(device)
        out.append(torch.softmax(model(pixel_values=x).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def read_frames(path, step):
    cap, k, frames = cv2.VideoCapture(path), 0, []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if k % step == 0:
            frames.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
        k += 1
    cap.release()
    return frames

still_prob = predict_pils([Image.open(p).convert('RGB') for p in chosen.path])
chosen['still_pred'] = [LABELS[i] for i in still_prob.argmax(1)]

clip_rows, frame_rows = [], []
for _, c in made_df.iterrows():
    frames = read_frames(f'{CLIPS_DIR}/{c['emotion']}/{c['clip']}', FRAME_STEP)
    if not frames:
        continue
    pr = predict_pils(frames)
    pred = pr.argmax(1)
    maj = np.bincount(pred, minlength=len(LABELS)).argmax()
    clip_rows.append({'clip': c['clip'], 'emotion': c.emotion, 'region': c.region, 'source_key': c.source_key,
                      'frames': len(pred), 'frame accuracy': (pred == lab2id[c.emotion]).mean(),
                      'majority': LABELS[maj], 'consistency': (pred == maj).mean(),
                      'still prediction': chosen.loc[chosen.source_key == c.source_key, 'still_pred'].iat[0]})
    for j, (pp, p) in enumerate(zip(pred, pr)):
        frame_rows.append({'clip': c['clip'], 'region': c.region, 'frame': j * FRAME_STEP, 'y': lab2id[c.emotion], 'pred': pp,
                           **dict(zip(LABELS, p))})
clips_df, frames_df = pd.DataFrame(clip_rows), pd.DataFrame(frame_rows)

summary = []
for region, fr in frames_df.groupby('region'):
    keys = set(clips_df[clips_df.region == region].source_key)
    st = chosen[chosen.source_key.isin(keys)]
    labels = sorted(set(st.label.map(lab2id)))
    f1_s = f1_score(st.label.map(lab2id), st.still_pred.map(lab2id), labels=labels, average='macro', zero_division=0)
    f1_f = f1_score(fr.y, fr.pred, labels=labels, average='macro', zero_division=0)
    cr = clips_df[clips_df.region == region]
    summary.append({'animation': region, 'clips': len(cr), 'source stills': len(st), 'frames': len(fr),
                    'macro-F1 stills': f1_s, 'macro-F1 frames': f1_f, 'delta macro-F1': f1_f - f1_s,
                    'mean consistency': cr.consistency.mean(), 'clips whose majority = true label': (cr.majority == cr.emotion).mean()})
summary = pd.DataFrame(summary).round(3)
display(summary)
display(clips_df.round(3))
summary.to_csv(f'{RESULTS_DIR}/animation_eval.csv', index=False)
clips_df.to_csv(f'{RESULTS_DIR}/animation_clips.csv', index=False)

## 5. Emotion timelines (per-frame probabilities)

In [ ]:
for region in clips_df.region.unique():
    show = clips_df[clips_df.region == region].groupby('emotion').head(1)
    fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.5 * len(show)), squeeze=False)
    for ax, (_, c) in zip(axes[:, 0], show.iterrows()):
        f = frames_df[frames_df['clip'] == c['clip']]
        for e in LABELS:
            ax.plot(f['frame'], f[e], label=e)
        ax.set_ylim(0, 1); ax.set_ylabel('probability')
        ax.set_title(f"[SYNTHETIC] {c['source_key']} - true: {c['emotion']} - animation: {region} - consistency {c['consistency']:.2f}", fontsize=9)
    axes[-1, 0].set_xlabel('frame'); axes[0, 0].legend(loc='upper right', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/emotion_timeline_{region}.png', dpi=150); plt.show()

## 6. Attention heatmaps on animated frames

In [ ]:
@torch.no_grad()
def attention_rollout(pil):
    x = tf(pil)[None].to(device)
    out = model(pixel_values=x, output_attentions=True)
    att = torch.stack(out.attentions).float().mean(2)
    a = att + torch.eye(att.size(-1), device=att.device)
    a = a / a.sum(-1, keepdim=True)
    roll = a[0]
    for l in range(1, a.size(0)):
        roll = a[l] @ roll
    m = roll[0, 0, 1:]
    s = int(m.numel() ** 0.5)
    m = (m / m.max()).reshape(s, s).cpu().numpy()
    return m, torch.softmax(out.logits.float(), -1)[0].cpu().numpy()

def overlay(rgb, mask, alpha=0.45):
    heat = cv2.applyColorMap((cv2.resize(mask, rgb.shape[1::-1]) * 255).astype(np.uint8), cv2.COLORMAP_JET)
    return (rgb * (1 - alpha) + heat[:, :, ::-1] * alpha).astype(np.uint8)

region = clips_df.region.iloc[0]
show = clips_df[clips_df.region == region].groupby('emotion').head(1)
fig, axes = plt.subplots(len(show), 4, figsize=(12, 3.2 * len(show)), squeeze=False)
for r_i, (_, c) in enumerate(show.iterrows()):
    frames = read_frames(f"{CLIPS_DIR}/{c['emotion']}/{c['clip']}", 1)
    for c_i, k in enumerate(np.linspace(0, len(frames) - 1, 4).astype(int)):
        pil = frames[k].resize((cfg['img_size'], cfg['img_size']))
        mask, p = attention_rollout(pil)
        axes[r_i, c_i].imshow(overlay(np.array(pil), mask)); axes[r_i, c_i].axis('off')
        axes[r_i, c_i].set_title(f"true {c['emotion']} | frame {k} | pred {LABELS[p.argmax()]} ({p.max():.2f})", fontsize=8)
plt.suptitle(f'Attention rollout on SYNTHETIC frames (animation: {region})'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/animation_attention.png', dpi=150); plt.show()
print('Saved to', RESULTS_DIR, ': animation_eval.csv, animation_clips.csv, emotion_timeline_*.png, animation_attention.png')